# Week 1 Polynomial & Interaction Terms

**Datasets:** UCL Online Retail II

## Topics
1. **Polynomial features (UCL)** — predict log monetary value from recency and log frequency with degree-2 terms.
2. **Interaction terms** — compare linear vs. polynomial test R² on held-out customers.


## Data loading and cleaning

Load UCL, Olist, and Taobao data for consistency with other weekly notebooks.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_orders_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_reviews_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_order_payments_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_Brazilian" / "olist_customers_dataset.csv")

pay = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    n_payments=("payment_sequential", "max"),
).reset_index()
itm = items.groupby("order_id").agg(
    total_price=("price", "sum"),
    freight_value=("freight_value", "sum"),
    n_items=("order_item_id", "count"),
).reset_index()
rev = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()

olist_df = (
    orders.merge(rev, on="order_id")
    .merge(itm, on="order_id")
    .merge(pay, on="order_id")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
)
olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_month"] = olist_df["order_purchase_timestamp"].dt.month
olist_df["delivery_days"] = (
    pd.to_datetime(olist_df["order_delivered_customer_date"], errors="coerce")
    - olist_df["order_purchase_timestamp"]
).dt.days
olist_df["delivery_days"] = olist_df["delivery_days"].fillna(olist_df["delivery_days"].median())
olist_df = olist_df.dropna(subset=["review_score", "payment_value", "total_price"])

ucl_raw = pd.read_excel(DATA_DIR / "ucl_dataset.xlsx")
ucl_raw = ucl_raw.rename(columns={"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"})
ucl_raw["InvoiceDate"] = pd.to_datetime(ucl_raw["InvoiceDate"])
ucl_raw = ucl_raw.dropna(subset=["CustomerID"])
ucl_raw = ucl_raw[~ucl_raw["InvoiceNo"].astype(str).str.startswith("C", na=False)]
ucl_raw = ucl_raw[(ucl_raw["Quantity"] > 0) & (ucl_raw["UnitPrice"] > 0)]
ucl_raw["LineTotal"] = ucl_raw["Quantity"] * ucl_raw["UnitPrice"]
anchor = ucl_raw["InvoiceDate"].max() + pd.Timedelta(days=1)
ucl_df = (
    ucl_raw.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda x: (anchor - x.max()).days),
        frequency=("InvoiceNo", "nunique"),
        monetary=("LineTotal", "sum"),
        avg_basket=("LineTotal", "mean"),
        n_items=("Quantity", "sum"),
    )
    .reset_index()
)
ucl_df["log_monetary"] = np.log1p(ucl_df["monetary"])
ucl_df["log_frequency"] = np.log1p(ucl_df["frequency"])

TAOBAO_PATH = DATA_DIR / "TaobaoUserBehavior.csv"
user_set = set()
chunks = []
for chunk in pd.read_csv(
    TAOBAO_PATH,
    header=None,
    names=["user_id", "item_id", "category_id", "behavior", "timestamp"],
    chunksize=500_000,
):
    chunk = chunk[chunk["user_id"].isin(user_set) | (len(user_set) < 50000)]
    if len(user_set) < 50000:
        for u in chunk["user_id"].unique():
            if len(user_set) >= 50000:
                break
            user_set.add(u)
    chunk = chunk[chunk["user_id"].isin(user_set)]
    if len(chunk):
        chunks.append(chunk)
    if len(user_set) >= 50000 and sum(len(c) for c in chunks) > 200_000:
        break

taobao = pd.concat(chunks, ignore_index=True)
taobao_df = (
    taobao.groupby("user_id")
    .agg(
        n_events=("behavior", "count"),
        n_pv=("behavior", lambda x: (x == "pv").sum()),
        n_cart=("behavior", lambda x: (x == "cart").sum()),
        n_fav=("behavior", lambda x: (x == "fav").sum()),
        n_buy=("behavior", lambda x: (x == "buy").sum()),
        n_categories=("category_id", "nunique"),
        n_items=("item_id", "nunique"),
    )
    .reset_index()
)
taobao_df["cart_rate"] = taobao_df["n_cart"] / taobao_df["n_events"].clip(lower=1)
taobao_df["converted"] = (taobao_df["n_buy"] > 0).astype(int)

print(f"Olist orders: {len(olist_df):,}")
print(f"UCL customers: {len(ucl_df):,}")
print(f"Taobao users: {len(taobao_df):,}")


# Linear regression baseline

Predict log monetary value from recency and log frequency.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_base = ucl_df[["recency", "log_frequency"]].values
y = ucl_df["log_monetary"].values
X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)

lin = LinearRegression().fit(X_train, y_train)
lin_r2 = r2_score(y_test, lin.predict(X_test))

print(f"Linear model test R²: {lin_r2:.4f}")
print("Coefficients:")
for name, coef in zip(["recency", "log_frequency"], lin.coef_):
    print(f"  {name}: {coef:.6f}")
print(f"  intercept: {lin.intercept_:.6f}")


# Polynomial regression (degree 2)

Add squared and interaction terms with standardized features.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

poly = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
poly.fit(X_train, y_train)
poly_r2 = r2_score(y_test, poly.predict(X_test))

coef_names = ["recency", "log_frequency", "recency^2", "recency*freq", "freq^2"]
coefs = poly.named_steps["model"].coef_

print(f"Polynomial model test R²: {poly_r2:.4f}")
print(f"Improvement over linear: {poly_r2 - lin_r2:.4f}")
print("\nStandardized coefficients:")
for name, coef in zip(coef_names, coefs):
    print(f"  {name}: {coef:.6f}")


Log transformation already captures most nonlinearity in spend. The polynomial model adds only a small test R² gain, so degree 2 is sufficient for this sample.


# Coefficient plot


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(coef_names, coefs, color="steelblue")
ax.set_xlabel("Coefficient (standardized features)")
ax.set_title("Week 1: Polynomial regression coefficients (UCL monetary value)")
plt.tight_layout()
plt.show()
